In [1]:
# =============================================================================
# 04_gold: Build the star schema (3 dimensions + 1 fact table)
# -----------------------------------------------------------------------------
# Input : output/silver/*.csv   (only rows with _is_valid = true are used)
# Output: output/gold/*.csv     dim_customers, dim_products, dim_sellers,
#                               fact_order_items
#
# Key design decisions
#   * Surrogate keys are a hash of the natural key (xxhash64, 63 bits), so the
#     same input always gives the same key on every run. No row_number().
#   * Fact foreign keys are taken by JOINING to the dimensions, so an orphan
#     key cannot exist. A fact row with no matching dimension row is written
#     to quarantine and counted (never dropped silently).
#   * Scope: dim_customers only contains customers who have an order on or after
#     2018-07-01 (Silver orders already hold only that range). dim_products and
#     dim_sellers are master data without an order date, so they are kept whole.
# =============================================================================

import os, glob, shutil
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import DecimalType

spark = (SparkSession.builder
         .master("local[*]")
         .appName("blue-owls-gold")
         .config("spark.sql.session.timeZone", "UTC")   # same time zone on every machine
         .config("spark.sql.shuffle.partitions", "8")   # small data: default 200 is wasteful
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

OUTPUT_DIR     = "/home/jovyan/work/output"
SILVER_DIR     = f"{OUTPUT_DIR}/silver"
GOLD_DIR       = f"{OUTPUT_DIR}/gold"
QUARANTINE_DIR = f"{OUTPUT_DIR}/quarantine"
for d in (GOLD_DIR, QUARANTINE_DIR):
    os.makedirs(d, exist_ok=True)

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"   # timestamp format used in the Silver CSVs
MONEY = DecimalType(12, 2)          # exact decimals for all money columns

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
# Records how many Silver rows were read and how many were valid (used in the report)
read_log = []


def read_silver(name, ts_cols=(), money_cols=(), int_cols=()):
    """
    Read one Silver CSV (all columns arrive as text), cast the typed columns back,
    and keep only valid rows. The number of rows ignored is logged, not hidden.
    """
    df = (spark.read.option("header", True).option("inferSchema", False)
          .csv(f"{SILVER_DIR}/{name}.csv"))
    for c in ts_cols:
        df = df.withColumn(c, F.to_timestamp(c, TS_FORMAT))
    for c in money_cols:
        df = df.withColumn(c, F.col(c).cast(MONEY))
    for c in int_cols:
        df = df.withColumn(c, F.col(c).cast("int"))
    df = df.withColumn("_is_valid", F.col("_is_valid") == "true")   # text -> boolean

    total = df.count()
    df = df.filter(F.col("_is_valid")).drop("_is_valid", "_invalid_reason", "_ingested_at")
    valid = df.count()
    read_log.append({"silver_table": name, "rows_in_silver": total,
                     "valid_rows_used": valid, "invalid_rows_ignored": total - valid})
    return df


def surrogate_key(*cols):
    """
    Deterministic surrogate key: a 63-bit hash of the natural key column(s).
    The same natural key always gives the same number, on every run and machine.
    The bit mask keeps the value positive. Uniqueness is checked in the validation cell.
    """
    return (F.xxhash64(*[F.col(c) for c in cols])
             .bitwiseAND(F.lit(0x7FFFFFFFFFFFFFFF)))


def write_single_csv(df, name, folder=GOLD_DIR):
    """
    Spark writes a folder of part files. We need ONE CSV per table, so we write one
    partition to a temp folder and move the part file to <name>.csv.
    mode=overwrite makes re-runs safe (same input -> same output, no duplicates).
    """
    tmp_dir = f"{folder}/_tmp_{name}"
    (df.coalesce(1).write.mode("overwrite")
       .option("header", True)
       .option("timestampFormat", TS_FORMAT)
       .csv(tmp_dir))
    part_file = glob.glob(f"{tmp_dir}/part-*.csv")[0]
    os.replace(part_file, f"{folder}/{name}.csv")
    shutil.rmtree(tmp_dir)

print("Helpers ready")

Helpers ready


In [3]:
orders = read_silver(
    "orders",
    ts_cols=["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"])

items = read_silver(
    "order_items",
    ts_cols=["shipping_limit_date"],
    money_cols=["price", "freight_value"],
    int_cols=["order_item_id"])

customers = read_silver("customers")
sellers   = read_silver("sellers")

products = read_silver(
    "products",
    money_cols=["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
    int_cols=["product_photos_qty", "product_description_length", "product_name_length"])

payments = read_silver(
    "payments",
    money_cols=["payment_value"],
    int_cols=["payment_sequential", "payment_installments"])

print(pd.DataFrame(read_log).to_string(index=False))

silver_table  rows_in_silver  valid_rows_used  invalid_rows_ignored
      orders           12824            12824                     0
 order_items           15590            14341                  1249
   customers           99441            99441                     0
     sellers            3095             3095                     0
    products           32951            32951                     0
    payments          103886            13225                 90661


In [4]:
# ---------------------------- dim_sellers -------------------------------------
dim_sellers = sellers.select(
    surrogate_key("seller_id").alias("seller_key"),
    "seller_id", "seller_city", "seller_state", "seller_zip_code_prefix")

# ---------------------------- dim_products ------------------------------------
# Volume = length x height x width. If any dimension is NULL the product is NULL,
# because multiplying with NULL gives NULL in Spark (exactly what the spec asks).
dim_products = products.select(
    surrogate_key("product_id").alias("product_key"),
    "product_id",
    F.coalesce(F.col("product_category_name"), F.lit("unknown")).alias("product_category_name"),
    "product_weight_g",
    (F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))
        .cast(DecimalType(18, 2)).alias("product_volume_cm3"),
    "product_photos_qty",
    "product_description_length")

# ---------------------------- dim_customers -----------------------------------
# 1) Spend per order = sum of (price + freight_value) over its items
order_spend = (items.groupBy("order_id")
               .agg(F.sum(F.col("price") + F.col("freight_value")).alias("order_spend")))

# 2) One row per order, enriched with the customer's attributes.
#    customer_id changes per order; customer_unique_id identifies the real person.
orders_c = (orders
            .join(customers.select("customer_id", "customer_unique_id", "customer_city",
                                   "customer_state", "customer_zip_code_prefix"),
                  on="customer_id", how="inner")
            .join(order_spend, on="order_id", how="left"))

# 3) City / state / zip come from the customer's MOST RECENT order.
#    order_id is a tie-breaker so the result is deterministic if two orders share a timestamp.
w_latest = (Window.partitionBy("customer_unique_id")
            .orderBy(F.col("order_purchase_timestamp").desc(), F.col("order_id").desc()))
latest = (orders_c.withColumn("_rn", F.row_number().over(w_latest))
          .filter(F.col("_rn") == 1)
          .select("customer_unique_id", "customer_city", "customer_state",
                  "customer_zip_code_prefix"))

# 4) Totals across all (in-scope) orders of the customer.
#    total_spend is 0 (not NULL) for a customer whose orders have no valid items.
agg = (orders_c.groupBy("customer_unique_id")
       .agg(F.to_date(F.min("order_purchase_timestamp")).alias("first_order_date"),
            F.countDistinct("order_id").cast("int").alias("total_orders"),
            F.coalesce(F.sum("order_spend"), F.lit(0)).cast(MONEY).alias("total_spend")))

dim_customers = (latest.join(agg, on="customer_unique_id", how="inner")
                 .select(surrogate_key("customer_unique_id").alias("customer_key"),
                         "customer_unique_id", "customer_city", "customer_state",
                         "customer_zip_code_prefix", "first_order_date", "total_orders",
                         "total_spend",
                         (F.col("total_orders") > 1).alias("is_repeat_customer")))

print("Dimensions built")

Dimensions built


In [5]:
# ---- Payment facts per order -------------------------------------------------
# Total paid and the max installment count over all payment rows of the order
pay_total = (payments.groupBy("order_id")
             .agg(F.sum("payment_value").alias("order_payment_total"),
                  F.max("payment_installments").alias("payment_installments")))

# payment_type = type of the payment row with the highest payment_value;
# on a tie, the first one alphabetically.
w_type = (Window.partitionBy("order_id")
          .orderBy(F.col("payment_value").desc(), F.col("payment_type").asc_nulls_last()))
pay_type = (payments.withColumn("_rn", F.row_number().over(w_type))
            .filter(F.col("_rn") == 1).select("order_id", "payment_type"))

pay = pay_total.join(pay_type, on="order_id", how="inner")

# ---- Items + order info + customer + payment info ----------------------------
# Inner join with orders: only items of valid, in-scope orders are kept.
# Left join with payments: an order without payments keeps its items (payment columns NULL).
base = (items
        .join(orders.select("order_id", "customer_id", "order_status", "order_purchase_timestamp",
                            "order_delivered_customer_date", "order_estimated_delivery_date"),
              on="order_id", how="inner")
        .join(customers.select("customer_id", "customer_unique_id"), on="customer_id", how="left")
        .join(pay, on="order_id", how="left"))

# ---- Split the order's payment across its items in proportion to price -------
w_order = Window.partitionBy("order_id")
w_rank  = Window.partitionBy("order_id").orderBy(F.col("price").desc(), F.col("order_item_id").asc())

base = (base
        .withColumn("_price_sum", F.sum("price").over(w_order))
        .withColumn("_n_items", F.count(F.lit(1)).over(w_order))
        # share of the order = price / total price (equal split if all prices are 0)
        .withColumn("_share", F.when(F.col("_price_sum") > 0,
                                     F.col("price").cast("double") / F.col("_price_sum").cast("double"))
                               .otherwise(1.0 / F.col("_n_items")))
        .withColumn("_rounded", F.round(F.col("order_payment_total").cast("double") * F.col("_share"), 2)
                                 .cast(MONEY))
        # Rounding can leave the split a few cents off the order total. The difference
        # goes to the most expensive item, so the shares add up exactly.
        .withColumn("_diff", F.col("order_payment_total") - F.sum("_rounded").over(w_order))
        .withColumn("_rank", F.row_number().over(w_rank))
        .withColumn("payment_value",
                    (F.col("_rounded") + F.when(F.col("_rank") == 1, F.col("_diff")).otherwise(F.lit(0)))
                     .cast(MONEY)))

# ---- Delivery metrics (all NULL when the order is not delivered) -------------
base = (base
        .withColumn("order_date", F.to_date("order_purchase_timestamp"))
        .withColumn("days_to_deliver",
                    F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp")))
        .withColumn("days_delivery_vs_estimate",
                    F.datediff(F.col("order_delivered_customer_date"), F.col("order_estimated_delivery_date")))
        # when(...) without otherwise() gives NULL when not delivered
        .withColumn("is_late_delivery",
                    F.when(F.col("days_delivery_vs_estimate").isNotNull(),
                           F.col("days_delivery_vs_estimate") > 0)))

# ---- Foreign keys: take them FROM the dimensions (so no orphan can exist) ----
keyed = (base
         .join(dim_customers.select("customer_unique_id", "customer_key"), on="customer_unique_id", how="left")
         .join(dim_products.select("product_id", "product_key"), on="product_id", how="left")
         .join(dim_sellers.select("seller_id", "seller_key"), on="seller_id", how="left")
         .withColumn("order_item_sk", surrogate_key("order_id", "order_item_id")))

# Rows that miss a dimension row are NOT dropped silently: counted and saved
missing_key = (F.col("customer_key").isNull() | F.col("product_key").isNull() | F.col("seller_key").isNull())
rejected = keyed.filter(missing_key)

fact_order_items = (keyed.filter(~missing_key)
                    .select("order_item_sk", "order_id", "order_item_id",
                            "customer_key", "product_key", "seller_key",
                            "order_date", "order_status", "price", "freight_value",
                            "payment_value", "payment_type", "payment_installments",
                            "days_to_deliver", "days_delivery_vs_estimate", "is_late_delivery"))

print("Fact table built")

Fact table built


In [6]:
# cache() so each table is computed once and not again for every write / check
gold = {
    "dim_customers":    dim_customers.cache(),
    "dim_products":     dim_products.cache(),
    "dim_sellers":      dim_sellers.cache(),
    "fact_order_items": fact_order_items.cache(),
}

for name, df in gold.items():
    write_single_csv(df, name)
    print(f"written: {GOLD_DIR}/{name}.csv")

# Save fact rows that could not be matched to every dimension (expected: 0)
n_rejected = rejected.count()
if n_rejected > 0:
    write_single_csv(rejected.select("order_id", "order_item_id", "customer_unique_id",
                                     "product_id", "seller_id"),
                     "fact_rejected_rows", folder=QUARANTINE_DIR)
print("fact rows rejected (missing dimension key):", n_rejected)

written: /home/jovyan/work/output/gold/dim_customers.csv
written: /home/jovyan/work/output/gold/dim_products.csv
written: /home/jovyan/work/output/gold/dim_sellers.csv
written: /home/jovyan/work/output/gold/fact_order_items.csv
fact rows rejected (missing dimension key): 0


In [7]:
def check(label, passed):
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")

print("===== RECORD COUNTS =====")
for name, df in gold.items():
    print(f"  {name:18s} {df.count():>8d}")

fact = gold["fact_order_items"]

print("\n===== FOREIGN KEYS (no orphan keys) =====")
for fk, dim in [("customer_key", "dim_customers"),
                ("product_key",  "dim_products"),
                ("seller_key",   "dim_sellers")]:
    orphans = fact.join(gold[dim].select(fk), on=fk, how="left_anti").count()
    check(f"every {fk} in fact_order_items exists in {dim} (orphans = {orphans})", orphans == 0)

print("\n===== UNIQUENESS =====")
for name, key in [("dim_customers", "customer_key"), ("dim_products", "product_key"),
                  ("dim_sellers", "seller_key"), ("fact_order_items", "order_item_sk")]:
    df = gold[name]
    check(f"{key} is unique in {name}", df.count() == df.select(key).distinct().count())
check("fact grain: (order_id, order_item_id) is unique",
      fact.count() == fact.select("order_id", "order_item_id").distinct().count())
check("dim_customers: one row per customer_unique_id",
      gold["dim_customers"].count() == gold["dim_customers"].select("customer_unique_id").distinct().count())

print("\n===== BUSINESS RULES =====")
# 1) Items of an order must add up to the order's payment total (to the cent)
recon = (fact.filter(F.col("payment_value").isNotNull())
         .groupBy("order_id").agg(F.sum("payment_value").alias("split_total"))
         .join(pay_total.select("order_id", "order_payment_total"), on="order_id")
         .filter(F.abs(F.col("split_total") - F.col("order_payment_total")) > 0.005))
check(f"payment split adds up to the order total (mismatching orders = {recon.count()})",
      recon.count() == 0)

# 2) is_late_delivery must be NULL exactly when the order is not delivered
bad_late = fact.filter(F.col("days_delivery_vs_estimate").isNull() != F.col("is_late_delivery").isNull()).count()
check(f"is_late_delivery is NULL only for undelivered orders (violations = {bad_late})", bad_late == 0)

# 3) No fact rows were lost along the way
check(f"fact rows + rejected rows = valid items in valid orders "
      f"({fact.count()} + {n_rejected})",
      fact.count() + n_rejected == items.join(orders.select("order_id"), "order_id").count())

print("\n===== FILES ON DISK (rows read back from the CSVs) =====")
for name, df in gold.items():
    csv_rows = spark.read.option("header", True).csv(f"{GOLD_DIR}/{name}.csv").count()
    check(f"{name}.csv rows = table rows ({csv_rows})", csv_rows == df.count())

print("\nSample of fact_order_items:")
spark.read.option("header", True).csv(f"{GOLD_DIR}/fact_order_items.csv").show(3, truncate=False)

===== RECORD COUNTS =====
  dim_customers         12647
  dim_products          32951
  dim_sellers            3095
  fact_order_items      14341

===== FOREIGN KEYS (no orphan keys) =====
  [PASS] every customer_key in fact_order_items exists in dim_customers (orphans = 0)
  [PASS] every product_key in fact_order_items exists in dim_products (orphans = 0)
  [PASS] every seller_key in fact_order_items exists in dim_sellers (orphans = 0)

===== UNIQUENESS =====
  [PASS] customer_key is unique in dim_customers
  [PASS] product_key is unique in dim_products
  [PASS] seller_key is unique in dim_sellers
  [PASS] order_item_sk is unique in fact_order_items
  [PASS] fact grain: (order_id, order_item_id) is unique
  [PASS] dim_customers: one row per customer_unique_id

===== BUSINESS RULES =====
  [PASS] payment split adds up to the order total (mismatching orders = 0)
  [PASS] is_late_delivery is NULL only for undelivered orders (violations = 0)
  [PASS] fact rows + rejected rows = valid item